# EQO notebook: inspect and evaluate OpenQEvo methods

Together with the Trotter notebook, this covers every admitted OpenQEvo operation: list methods, inspect method context, evaluate a bounded dense reference, and synthesize a circuit. This notebook uses only published EQO workflows; it does not import OpenQEvo directly.

In [ ]:
import json
import os

from eqo import EQOClient, render_artifact, render_run

eqo = EQOClient.connect(os.environ.get("EQO_ENDPOINT", "http://127.0.0.1:8080"))
published = {item["id"]: item for item in eqo.workflows.list()}
eqo.health()

## List the registered methods

The result records the exact OpenQEvo source and local runtime admitted by EQO.

In [ ]:
catalog = published.get("openqevo-method-catalog")
if catalog is None:
    raise RuntimeError("openqevo-method-catalog is not published by this EQO profile.")
catalog_run = eqo.workflows.submit(catalog["id"], catalog["version"])
catalog_done = catalog_run.wait(timeout=120)
if catalog_done.state != "succeeded":
    raise RuntimeError(f"Method catalog ended in {catalog_done.state}.")
render_artifact(catalog_done.artifacts.by_type("qhpc.method-catalog@1"))

## Inspect one method's context

This is separate from a numerical or circuit run: it returns documented applicability, limitations, complexity, and references for the pinned `trotter_s2` method.

In [ ]:
context_workflow = published.get("openqevo-method-context")
if context_workflow is None:
    raise RuntimeError("Restart EQO Local to publish openqevo-method-context.")
context_run = eqo.workflows.submit(context_workflow["id"], context_workflow["version"])
context_done = context_run.wait(timeout=120)
if context_done.state != "succeeded":
    raise RuntimeError(f"Method context ended in {context_done.state}.")
render_artifact(context_done.artifacts.by_type("qhpc.evolution-method-context@1"))

## Evaluate a bounded dense reference

The dense result is deliberately limited to small systems. It is a numerical reference, not a scalable solver, circuit, or hardware result.

In [ ]:
hamiltonian = {
    "qubits": 2,
    "terms": [
        {"pauli": "ZI", "coefficient": 1.0},
        {"pauli": "IZ", "coefficient": 0.5},
    ],
}
input_hamiltonian = eqo.artifacts.create_input(
    "qhpc.pauli-hamiltonian@1", json.dumps(hamiltonian), name="reference-hamiltonian.json"
)
dense = published.get("openqevo-dense-reference")
if dense is None:
    raise RuntimeError("openqevo-dense-reference is not published by this EQO profile.")
dense_run = eqo.workflows.submit(
    dense["id"], dense["version"], inputs={"hamiltonian": input_hamiltonian.id}
)
render_run(dense_run)

In [ ]:
dense_done = dense_run.wait(timeout=300)
if dense_done.state != "succeeded":
    raise RuntimeError(f"Dense reference ended in {dense_done.state}; inspect render_run(dense_done).")
display(render_artifact(dense_done.artifacts.by_type("qhpc.evolution-result@1")))
display(render_artifact(dense_done.artifacts.by_type("qhpc.dense-unitary@1")))